# PNCP — Identificação de Subenquadramento de Engenharia

TCC MBA IA & Big Data (ICMC/USP) — Profa. Solange O. Rezende

**Como usar:** rode as células em ordem. Cada etapa **persiste em disco**
(parquet/json/npy), então se o kernel reiniciar você continua de onde parou.

## Célula 1 — Setup (rode 1× por sessão)

In [ ]:
!pip install -q wordcloud tqdm imbalanced-learn nltk mlxtend psutil
!pip install -q pymupdf pdfplumber pytesseract networkx python-louvain openpyxl
!pip install -q sentence-transformers transformers torch statsmodels
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por > /dev/null
print('OK — dependências instaladas')

## Célula 2 — Drive + clonar o repositório

In [ ]:
import sys, os
if not os.path.exists('/content/pncp-engineering-analytics'):
    !git clone https://github.com/caiofvserra/pncp-engineering-analytics.git /content/pncp-engineering-analytics
sys.path.insert(0, '/content/pncp-engineering-analytics')

import pncp
pncp.montar_drive('/content/drive/MyDrive/PNCP_TCC')
pncp.keep_alive()
pncp.ram.monitorar_ram('início')

## Célula 3 — Coleta (escreve parquet por ano, libera RAM entre anos)

Para SP/2023–2025: ~3 minutos por ano (200–300k contratos no total).

In [ ]:
pncp.coleta.coletar(uf='SP', anos=range(2023, 2026), tamanho=500)
pncp.ram.monitorar_ram('após coleta')

## Célula 4 — EDA + pré-processamento de texto + TF-IDF

In [ ]:
pncp.eda.executar()
from pncp import config
caminho = config.caminho(config.SUB_COLETA, 'contratos.parquet')
pncp.texto.preprocessar(caminho)
pncp.texto.marcar_termos_dominio(caminho)
pncp.texto.construir_tfidf(caminho)
pncp.ram.monitorar_ram('após texto/TF-IDF')

## Célula 5 — Classificação supervisionada (LR + RF + SVC)

Inclui: holdout estratificado, GridSearch (LR), McNemar, bootstrap, ranking de suspeitos.

In [ ]:
pncp.classificacao.executar(
    fazer_grid=True,
    fazer_holdout=True,
    fazer_mcnemar=True,
    fazer_bootstrap=True,
)
pncp.ram.monitorar_ram('após classificação')

## Célula 6 — Técnicas avançadas (LDA, Label Propagation, Apriori, KMeans, Hierárquico)

In [ ]:
pncp.avancado.executar(
    fazer_lda=True,
    fazer_lp=True,
    fazer_apriori=True,
    fazer_kmeans=True,
    fazer_hier=True,
)
pncp.ram.monitorar_ram('após avançado')

## Célula 7 — Embeddings BERTimbau (opcional, mais lento)

Use `tipo='sentence-bert'` para iterar rápido; `'bertimbau'` para versão final do TCC.

In [ ]:
# pncp.embeddings.executar(tipo='sentence-bert', treinar=True)
# pncp.ram.monitorar_ram('após embeddings')

## Célula 8 — Camadas externas (PDFs, Aditivos, Grafos, CNAE)

In [ ]:
pncp.pdfs.executar(max_contratos=200)
pncp.aditivos.executar(max_contratos=200)
pncp.grafos.executar()
pncp.cnae.executar(
    max_consultas=200,
    caminho_excel_crea='/content/drive/MyDrive/PNCP_TCC/cnaes_crea.xlsx',
)
pncp.ram.monitorar_ram('após camadas externas')

## Célula 9 — Relatório final TCC + consolidação

In [ ]:
pncp.relatorio.gerar()
from pathlib import Path
rel = Path(pncp.config.PASTA_DADOS) / pncp.config.SUB_P9 / 'relatorio.md'
print(rel.read_text(encoding='utf-8'))

## Célula 10 — Validação contra ground truth manual (opcional)

1. Abra `dados/cnae/amostra_revisao_manual.csv`
2. Preencha a coluna `revisao_manual`: `subenq` / `ok` / `duv`
3. Rode esta célula

In [ ]:
csv = pncp.config.caminho(pncp.config.SUB_P8, 'amostra_revisao_manual.csv')
pncp.relatorio.validar_ground_truth(csv)

## Glossário

Não lembra um termo? Rode `pncp.relatorio.glossario()` para ver tudo,
ou `pncp.relatorio.glossario('F1')` para um termo específico.

In [ ]:
pncp.relatorio.glossario()